<a href="https://colab.research.google.com/github/myresearchbvp/ERP-MCDA-Simulation/blob/main/Automated_ERP_PreSelection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# @title
# -*- coding: utf-8 -*-
"""Automated_ERP_PreSelection.ipynb

Automatically generated by Colab.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from google.colab import files
import copy

# === 1. DATA & INITIALIZATION ===
criteria = ["Functional Coverage", "Feasibility", "Scalability", "Usability", "Integration", "Vendor Support"]
extended_alts = [f"ERP-{chr(65+i)}" for i in range(10)]

original_scores = {
    "ERP-A": [5, 3, 4, 4, 4, 5], "ERP-B": [4, 4, 3, 3, 4, 4],
    "ERP-C": [3, 5, 3, 4, 3, 3], "ERP-D": [4, 3, 5, 3, 4, 4]
}
for i in range(4, 10): original_scores[f"ERP-{chr(65+i)}"] = [3, 3, 3, 3, 3, 3]

working_scores = {k: list(v) for k, v in original_scores.items()}

presets = {
    "S1": [0.24, 0.16, 0.18, 0.14, 0.16, 0.12],
    "S2": [0.20, 0.25, 0.15, 0.14, 0.14, 0.12],
    "S3": [0.20, 0.12, 0.28, 0.12, 0.16, 0.12]
}
presets["S-Avg"] = [round((presets["S1"][i] + presets["S2"][i] + presets["S3"][i]) / 3, 3) for i in range(6)]

# === 2. UI WIDGETS ===
slider_erps = widgets.IntSlider(value=4, min=2, max=10, description='ERPs:')

sliders_w = [widgets.FloatSlider(value=v, min=0, max=1, step=0.01, description=c,
                                 style={'description_width': 'initial'}, layout={'width': '350px'})
             for v, c in zip(presets["S1"], criteria)]

weight_sum_label = widgets.HTML(value="")

btn_s1 = widgets.Button(description="Baseline (S1)", button_style='info')
btn_s2 = widgets.Button(description="Feasibility (S2)", button_style='info')
btn_s3 = widgets.Button(description="Growth (S3)", button_style='info')
btn_s_avg = widgets.Button(description="Consensus (S-Avg)", button_style='primary')
scenario_desc = widgets.HTML("<i style='font-size:12px; color:gray;'>S1: Baseline balanced needs.</i>")

erp_selector = widgets.Dropdown(options=extended_alts[:4], description='Edit ERP:',
                                style={'description_width': 'initial'}, layout={'width': '180px'})

score_inputs = [widgets.BoundedIntText(value=3, min=1, max=5, layout={'width': '50px'}) for _ in range(6)]
btn_apply_scores = widgets.Button(description="Apply", button_style='warning', layout={'width': '80px'})

btn_export_txt = widgets.Button(description="Download report (.txt)", button_style='success', icon='file-text', layout={'width': '220px'})
btn_export_png = widgets.Button(description="Download charts (.png)", button_style='primary', icon='image', layout={'width': '220px'})
btn_export_csv = widgets.Button(description="Download data (.csv)", button_style='info', icon='table', layout={'width': '220px'})
btn_sens = widgets.Button(description="Run sensitivity analysis", button_style='warning', icon='line-chart', layout={'width': '220px'})
btn_reset = widgets.Button(description="Reset to original data", button_style='danger', icon='undo', layout={'width': '220px'})

out = widgets.Output()
last_report_txt = ""
global_fig = None
current_w_norm = []

# === 3. CORE LOGIC ===
def check_pareto_dominance(winner, runner, scores_dict):
    w_scores = np.array(scores_dict[winner])
    r_scores = np.array(scores_dict[runner])
    if np.all(w_scores >= r_scores) and np.any(w_scores > r_scores): return True
    return False

def get_strengths_weaknesses(winner, ctb_dict):
    w_ctb = ctb_dict[winner]
    return criteria[np.argmax(w_ctb)], criteria[np.argmin(w_ctb)]

def get_head_to_head(top1, top2, scores_dict):
    s1, s2 = scores_dict[top1], scores_dict[top2]
    w, l, t = 0, 0, 0
    for v1, v2 in zip(s1, s2):
        if v1 > v2: w += 1
        elif v1 < v2: l += 1
        else: t += 1
    return w, l, t

def get_consistency(winner, scores_dict):
    std_dev = np.std(scores_dict[winner])
    if std_dev <= 0.85: return "High (balanced and predictable)"
    elif std_dev <= 1.25: return "Moderate (some variations)"
    else: return "Low (risky, has extreme highs and lows)"

def run_mcda(w, current_scores, alt_list):
    sc, ctb = {}, {}
    for a in alt_list:
        s_raw = np.array(current_scores[a])
        w_arr = np.array(w)
        c_vals = s_raw * w_arr
        ctb[a] = c_vals
        sc[a] = float(np.sum(c_vals))

    rk = sorted(sc.items(), key=lambda x: x[1], reverse=True)
    max_s = rk[0][1]
    winners = [item[0] for item in rk if abs(item[1] - max_s) < 1e-7]
    remaining = [item for item in rk if item[0] not in winners]
    runners, margin = ([], 0.0)
    if remaining:
        max_run = remaining[0][1]
        runners = [item[0] for item in remaining if abs(item[1] - max_run) < 1e-7]
        margin = max_s - max_run
    return rk, winners, runners, margin, ctb

def get_top_entities(rk, top_n=3):
    if not rk: return []
    scores = sorted(list(set([v for k, v in rk])), reverse=True)[:top_n]
    return [k for k, v in rk if v in scores]

def format_report_txt(rk, w_norm, wins, runs, margin, ctb, is_pareto, show_sens=False, stability_txt=""):
    report = "ERP SELECTION SYSTEM - DECISION REPORT\n" + "="*40 + "\n\n1. CRITERIA WEIGHTS (See Chart 3 for visual distribution):\n"
    for c, w in zip(criteria, w_norm): report += f"   - {c}: {w*100:.1f}%\n"

    if np.allclose(w_norm, presets["S-Avg"], atol=0.01):
        report += f"   * CONSENSUS MODE: These weights represent a mathematical average of all predefined scenarios.\n"

    report += "\n2. FINAL RANKING:\n"
    for i, (a, s) in enumerate(rk): report += f"   {i+1}. {a}: {s:.3f}\n"

    report += "\n3. ANALYSIS & IN-DEPTH METRICS:\n"
    if len(wins) > 1:
        report += f"   Technical tie between: {', '.join(wins)}\n"
        report += f"   * ROBUSTNESS: Inconclusive due to tie.\n"
    else:
        best_c, worst_c = get_strengths_weaknesses(wins[0], ctb)
        efficiency = (rk[0][1] / 5.0) * 100
        consistency = get_consistency(wins[0], working_scores)

        report += f"   Recommended: {wins[0]}\n"
        report += f"   Margin: {margin:.3f} points over {runs[0] if runs else 'N/A'}\n"

        if margin < 0.1:
            report += f"   * ROBUSTNESS: Low (volatile). The {margin:.3f} margin is small; minor priority changes could alter the winner.\n"
        else:
            report += f"   * ROBUSTNESS: High (stable). The {margin:.3f} margin provides a comfortable lead resilient to minor changes.\n"

        report += f"   Strengths: {best_c} | Weakest link: {worst_c}\n"
        report += f"   Efficiency to ideal solution: {efficiency:.1f}% (TOPSIS concept: closeness to a hypothetical perfect system scoring 5/5 on all criteria)\n"
        report += f"   Performance consistency: {consistency} (Risk indicator based on standard deviation of raw scores)\n"

        if runs:
            hw, hl, ht = get_head_to_head(wins[0], runs[0], working_scores)
            report += f"   Head-to-head vs {runs[0]}: Wins in {hw}, loses in {hl} and ties in {ht} criteria.\n"

            best_w_idx = np.argmax(w_norm)
            pts_needed = margin / w_norm[best_w_idx]
            report += f"   Gap analysis (goal seeking): {runs[0]} needs approx. +{pts_needed:.1f} raw points in '{criteria[best_w_idx]}' to overtake the winner.\n"

        if is_pareto:
            report += f"   * PARETO STATUS: {wins[0]} strictly dominates its runner-up. Decision is mathematically incontestable.\n"
        else:
            report += f"   * PARETO STATUS: No strict dominance. Trade-offs exist.\n"

    if stability_txt:
        report += f"\n4. CROSS-SCENARIO STABILITY (vs Baseline S1):\n   {stability_txt}\n"

    if show_sens and len(rk) >= 2:
        top1 = rk[0][0]
        best_c_idx = np.argmax(ctb[top1])
        report += f"\n5. SENSITIVITY ANALYSIS (See Chart 4):\n"
        report += f"   Simulated a ±20% weight variation on the winner's most impactful criterion ('{criteria[best_c_idx]}').\n"
        report += f"   (Tests if the decision is stable. If lines cross in the chart, the winner changes under different priorities).\n"

    return report

# === 4. VISUALIZATION ===
def plot_results(ctb, alt_list, winners, rk, show_sens=False):
    global global_fig

    n_rows = 2 if show_sens else 1
    fig = plt.figure(figsize=(15, 11))
    global_fig = fig

    # --- CHART 1: Bar Chart ---
    ax1 = plt.subplot(2, 2, 1)
    df = pd.DataFrame(ctb, index=criteria).T.loc[alt_list]
    df.plot(kind='barh', stacked=True, ax=ax1, colormap='viridis', edgecolor='white')

    for i, (name, total) in enumerate(df.sum(axis=1).items()):
        ax1.text(total + 0.05, i, f'{total:.3f}', va='center', fontweight='bold')
        if name in winners:
            for patch in ax1.patches:
                if abs(patch.get_y() - (i - 0.25)) < 0.1:
                    patch.set_linewidth(2.5); patch.set_edgecolor('black')

    ax1.set_title("Chart 1: Weighted scoring breakdown", pad=20, fontweight='bold')
    ax1.set_xlabel("Aggregate score")
    ax1.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3)
    ax1.grid(axis='x', alpha=0.3)

    # --- CHART 2: Radar Chart ---
    ax2 = plt.subplot(2, 2, 2, polar=True)
    top_alts = get_top_entities(rk, 3)

    angles = np.linspace(0, 2 * np.pi, len(criteria), endpoint=False).tolist()
    angles += angles[:1]

    max_val = 0
    lines, labels = [], []
    linestyles = ['-', '--', ':']
    markers = ['o', 's', '^']

    for idx, a in enumerate(top_alts):
        vals = list(ctb[a])
        vals += vals[:1]
        max_val = max(max_val, max(vals))
        ls = linestyles[idx % len(linestyles)]
        mk = markers[idx % len(markers)]

        l, = ax2.plot(angles, vals, linewidth=2.5, linestyle=ls, marker=mk, markersize=7, alpha=0.85)
        ax2.fill(angles, vals, alpha=0.08)
        lines.append(l)
        labels.append(a)

    ax2.set_xticks(angles[:-1])
    ax2.set_xticklabels(criteria, size=9)
    ax2.set_ylim(0, max_val * 1.15)
    ax2.set_title("Chart 2: Weighted score profile (top performers)", pad=20, fontweight='bold')
    ax2.legend(lines, labels, loc='upper right', bbox_to_anchor=(1.3, 1.1))

    # --- CHART 3: Donut Chart ---
    ax3 = plt.subplot(2, 2, 3)
    pastel_colors = plt.cm.Set3(np.linspace(0, 1, len(criteria)))
    wedges, texts, autotexts = ax3.pie(current_w_norm, labels=criteria, autopct='%1.1f%%', startangle=140,
                                       colors=pastel_colors,
                                       wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2))

    plt.setp(autotexts, size=10, weight="bold", color="black")
    plt.setp(texts, size=9)
    ax3.set_title("Chart 3: Active criteria weights distribution", pad=20, fontweight='bold')

    # --- CHART 4: Sensitivity ---
    if show_sens and len(rk) >= 2:
        ax4 = plt.subplot(2, 2, 4)
        top1, top2 = rk[0][0], rk[1][0]

        best_c_idx = np.argmax(ctb[top1])
        variations = np.linspace(-0.2, 0.2, 5)
        scores_t1, scores_t2 = [], []
        x_labels = []

        for v in variations:
            w_test = list(current_w_norm)
            w_test[best_c_idx] = max(0, min(1, w_test[best_c_idx] + v))
            w_test_sum = sum(w_test)
            w_test = [wt/w_test_sum for wt in w_test]

            sc_t1 = sum(np.array(working_scores[top1]) * np.array(w_test))
            sc_t2 = sum(np.array(working_scores[top2]) * np.array(w_test))

            scores_t1.append(sc_t1)
            scores_t2.append(sc_t2)
            x_labels.append(f"{v*100:+.0f}%")

        ax4.plot(x_labels, scores_t1, marker='s', label=f"{top1} (Winner)", linewidth=2.5)
        ax4.plot(x_labels, scores_t2, marker='o', label=f"{top2} (Runner-up)", linewidth=2.5, linestyle='--')

        ax4.set_title(f"Chart 4: Sensitivity analysis on '{criteria[best_c_idx]}'", pad=15, fontweight='bold')
        ax4.set_xlabel("Weight variation (simulates tests for decision robustness)")
        ax4.set_ylabel("Total score")
        ax4.grid(alpha=0.4)
        ax4.legend()

    plt.tight_layout()
    plt.show()

# === 5. HANDLERS ===
def update_dashboard(show_sensitivity=False):
    global last_report_txt, current_w_norm
    n = slider_erps.value
    current_alts = extended_alts[:n]

    if erp_selector.options != tuple(current_alts): erp_selector.options = current_alts

    w_raw = [sl.value for sl in sliders_w]
    w_sum = sum(w_raw)

    # FIX: Tightened the tolerance to 0.002.
    # This safely absorbs 0.999 from S-Avg but triggers the warning for 1.01 or 0.99.
    if abs(w_sum - 1.0) <= 0.002:
        weight_sum_label.value = f"<span style='color:green; font-size:13px;'>Sum: <b>{w_sum:.2f}</b> (Perfect)</span>"
    else:
        weight_sum_label.value = f"<span style='color:#d35400; font-size:13px;'>Sum: <b>{w_sum:.2f}</b> (Warning: The system automatically normalizes weights to 100% in the background, but for methodological clarity, we advise adjusting sliders to equal exactly 1.0).</span>"

    w_norm = [w/w_sum for w in w_raw] if w_sum > 0 else [1/6]*6
    current_w_norm = w_norm

    rk, wins, runs, margin, ctb = run_mcda(w_norm, working_scores, current_alts)

    is_pareto = False
    pareto_html = ""
    swot_html = ""
    h2h_html = ""
    robustness_html = ""
    s_avg_html = ""
    gap_html = ""
    sens_html = ""
    stability_html = ""
    stability_txt_report = ""

    if np.allclose(w_norm, presets["S-Avg"], atol=0.01):
        s_avg_html = "<div style='background-color:#e8eaf6; color:#283593; padding:12px; border-left:5px solid #3f51b5; margin-bottom:15px;'>🤝 <b>Consensus mode active (S-Avg):</b> The weights now applied are the exact mathematical average of scenarios S1, S2 and S3.<br><small><i>(Simulates a democratic compromise when different stakeholders have conflicting priorities. See <b>Chart 3</b> above to visualize this combined weight distribution).</i></small></div>"

    # --- AUTOMATED CROSS-SCENARIO STABILITY ANALYZER ---
    baseline_w = presets["S1"]
    scores_modified = any(working_scores[k] != original_scores[k] for k in current_alts)

    if scores_modified:
        if not np.allclose(w_norm, baseline_w, atol=0.01):
            stability_html = f"<div style='background-color:#fff3cd; color:#856404; padding:12px; border-left:5px solid #ffc107; margin-bottom:15px;'>⚠️ <b>Automated Stability explanation Disabled:</b> Raw scores have been manually modified. The stability tracking module mathematically requires a constant performance matrix to isolate the effect of weight changes. Please click 'Reset to original data' to re-enable cross-scenario tracking.</div>"
    else:
        if not np.allclose(w_norm, baseline_w, atol=0.01):
            rk_b, wins_b, runs_b, margin_b, ctb_b = run_mcda(baseline_w, working_scores, current_alts)
            if len(wins_b) == 1 and len(wins) == 1:
                if wins_b[0] == wins[0]:
                    margin_diff = margin - margin_b
                    trend = "expansion" if margin_diff > 0 else "compression"
                    delta_c = [(w_norm[i] - baseline_w[i]) * working_scores[wins_b[0]][i] for i in range(len(criteria))]
                    driver_idx = np.argmax(np.abs(delta_c))
                    driver_crit = criteria[driver_idx]

                    stability_txt_report = f"When shifting from the Baseline scenario to the current weights, the ranking remains stable with {wins_b[0]} keeping the lead. However, the system detects a margin {trend} (from {margin_b:.3f} to {margin:.3f}). According to the rule-based diagnostic, this shift is primarily driven by the weight change in '{driver_crit}', interacting with the winner's raw score."
                    stability_html = f"<div style='background-color:#e0f2f1; color:#004d40; padding:12px; border-left:5px solid #009688; margin-bottom:15px;'>🔍 <b>Automated Stability explanation (vs Baseline S1):</b><br>{stability_txt_report}</div>"
                else:
                    stability_txt_report = f"RANK REVERSAL DETECTED! Under the Baseline scenario, {wins_b[0]} was the winner. Under the current weights, {wins[0]} takes the lead. This indicates high sensitivity to the modified priorities."
                    stability_html = f"<div style='background-color:#ffebee; color:#b71c1c; padding:12px; border-left:5px solid #f44336; margin-bottom:15px;'>⚠️ <b>Automated Stability explanation (vs Baseline S1):</b><br>{stability_txt_report}</div>"

    if show_sensitivity and len(rk) >= 2:
        sens_html = f"<div style='background-color:#f3e5f5; color:#4a148c; padding:12px; border-left:5px solid #9c27b0; margin-bottom:15px;'>📈 <b>Sensitivity analysis active:</b> Simulated a ±20% weight variation on the winner's top criterion (see <b>Chart 4</b>).<br><small><i>(Tests if the decision is mathematically stable. If the lines in Chart 4 cross, it means the runner-up would win if priorities shift slightly. If they don't cross, your choice is highly robust).</i></small></div>"

    if len(wins) == 1 and len(runs) > 0:
        is_pareto = check_pareto_dominance(wins[0], runs[0], working_scores)
        best_c, worst_c = get_strengths_weaknesses(wins[0], ctb)
        efficiency = (rk[0][1] / 5.0) * 100
        consistency = get_consistency(wins[0], working_scores)

        if margin < 0.1:
            robustness_html = f"<div style='background-color:#fff3cd; color:#856404; padding:12px; border-left:5px solid #ffc107; margin-bottom:15px;'>🛡️ <b>Decision robustness: Low (volatile).</b> The point margin ({margin:.3f}) is very small. A slight change in priority weights could easily flip the ranking. Proceed with caution.</div>"
        else:
            robustness_html = f"<div style='background-color:#d1ecf1; color:#0c5460; padding:12px; border-left:5px solid #17a2b8; margin-bottom:15px;'>🛡️ <b>Decision robustness: High (stable).</b> The winner maintains a comfortable lead (margin of {margin:.3f}). The decision is highly resilient to minor priority changes.</div>"

        best_w_idx = np.argmax(w_norm)
        pts_needed = margin / w_norm[best_w_idx]
        gap_html = f"<br><br>🎯 <b>Gap analysis (goal seeking):</b> For <b>{runs[0]}</b> to overtake the winner, it would need to improve its raw score in its highest-weighted criterion (<b>{criteria[best_w_idx]}</b>) by approximately <b>+{pts_needed:.1f}</b> points.<br><small><i>(Calculates the exact raw performance improvement required for the runner-up to change the final decision).</i></small>"

        swot_html = f"<br><br>🔹 <b>Strength:</b> {best_c} | 🔸 <b>Weakness:</b> {worst_c}<br><br>🌟 <b>Efficiency to ideal solution:</b> {efficiency:.1f}% <br><small><i>(Inspired by the TOPSIS method. It measures how close the recommended system is to a hypothetical perfect ERP that scores a maximum of 5 on all criteria. A score of 100% indicates absolute perfection).</i></small><br><br>⚖️ <b>Performance consistency:</b> {consistency}<br><small><i>(Analyzes the standard deviation of raw scores. Consistent scores [e.g., 4, 4, 4] are predictable and safe, while fluctuating scores [e.g., 5, 2, 5] represent a higher implementation risk).</i></small>{gap_html}"

        hw, hl, ht = get_head_to_head(wins[0], runs[0], working_scores)
        h2h_html = f"<div style='background-color:#e8f4f8; color:#0c5460; padding:12px; border-left:5px solid #17a2b8; margin-bottom:15px;'>🥊 <b>Head-to-head outranking:</b> <b>{wins[0]}</b> vs <b>{runs[0]}</b><br>Out of {len(criteria)} criteria, the winner directly beats the runner-up in <b>{hw}</b>, loses in <b>{hl}</b> and ties in <b>{ht}</b>.<br><small><i>(Evaluates direct pairwise performance and ignores the weights. It simply counts in how many criteria the winner is better, worse or equal to the runner-up. Concept derived from PROMETHEE/ELECTRE methods).</i></small></div>"

        if is_pareto:
            pareto_html = f"<div style='background-color:#d4edda; color:#155724; padding:12px; border-left:5px solid #28a745; margin-bottom:15px;'>🏆 <b>Pareto dominance analysis:</b> <b>{wins[0]}</b> strictly dominates <b>{runs[0]}</b>.<br><small><i>(The winner scored higher or equal in every single raw criterion compared to its closest rival. This makes the ranking mathematically incontestable regardless of how you change the weights. See <b>Chart 2</b> to visualize).</i></small></div>"
        else:
            pareto_html = f"<div style='background-color:#e2e3e5; color:#383d41; padding:12px; border-left:5px solid #6c757d; margin-bottom:15px;'>⚖️ <b>Pareto dominance analysis:</b> No strict dominance. <b>{wins[0]}</b> and <b>{runs[0]}</b> have trade-offs.<br><small><i>(To achieve Pareto dominance, the winning system must score higher or equal in every single criterion compared to its closest rival).</i></small></div>"
    elif len(wins) > 1:
        robustness_html = f"<div style='background-color:#e2e3e5; color:#383d41; padding:12px; border-left:5px solid #6c757d; margin-bottom:15px;'>🛡️ <b>Decision robustness: Inconclusive.</b> There is a technical tie. Refine weights or criteria.</div>"
        pareto_html = f"<div style='background-color:#e2e3e5; color:#383d41; padding:12px; border-left:5px solid #6c757d; margin-bottom:15px;'>⚖️ <b>Pareto dominance analysis:</b> Cannot be calculated (technical tie for 1st place).</div>"

    last_report_txt = format_report_txt(rk, w_norm, wins, runs, margin, ctb, is_pareto, show_sens=show_sensitivity, stability_txt=stability_txt_report)

    with out:
        clear_output(wait=True)
        plot_results(ctb, current_alts, wins, rk, show_sens=show_sensitivity)

        if s_avg_html: display(HTML(s_avg_html))
        if stability_html: display(HTML(stability_html))
        if sens_html: display(HTML(sens_html))
        display(HTML(robustness_html))
        display(HTML(pareto_html))
        display(HTML(h2h_html))

        if len(wins) > 1:
            display(HTML(f"<div style='background-color:#fff3cd; padding:15px; border-left:5px solid #ffc107;'><b>Decision:</b> Technical tie between {', '.join(wins)} (Score: {rk[0][1]:.3f}).<br><small>Advice: Adjust weights to break the tie.</small></div>"))
        else:
            display(HTML(f"<div style='background-color:#f8f9fa; padding:15px; border-left:5px solid #007bff;'><b>Recommendation:</b> {wins[0]} is the optimal choice (Score: {rk[0][1]:.3f}). Leads over {runs[0] if runs else 'N/A'} by {margin:.3f} points.{swot_html}</div>"))

        display(HTML("<br><b>Current raw scores (heatmap verification table):</b><br><small><i>(Darker green highlights stronger performance areas)</i></small>"))
        df_table = pd.DataFrame({c: [working_scores[a][i] for a in current_alts] for i, c in enumerate(criteria)}, index=current_alts)
        styled_df = df_table.style.background_gradient(cmap='Greens', axis=None, vmin=1, vmax=5).format("{:.0f}")
        display(styled_df)

def on_erp_change(change):
    for i, val in enumerate(working_scores[change['new']]): score_inputs[i].value = val

def load_scenario(b):
    if "S-Avg" in b.description:
        key = "S-Avg"
        desc_text = "Applied S-Avg: Mathematical consensus of S1, S2 and S3 (See Chart 3 for distribution)."
    else:
        key = b.description.split('(')[1].split(')')[0]
        desc_texts = {"S1": "Baseline balanced needs.", "S2": "Feasibility focus.", "S3": "Growth focus."}
        desc_text = f"Applied {key}: {desc_texts.get(key, '')} (See Chart 3 for distribution)."

    for sl, val in zip(sliders_w, presets[key]): sl.value = val
    scenario_desc.value = f"<i style='font-size:12px; color:gray;'>{desc_text}</i>"

def reset_scores(b):
    global working_scores
    working_scores = {k: list(v) for k, v in original_scores.items()}

    slider_erps.value = 4
    erp_selector.value = 'ERP-A'
    for i, val in enumerate(working_scores['ERP-A']): score_inputs[i].value = val
    for sl, val in zip(sliders_w, presets["S1"]): sl.value = val
    scenario_desc.value = "<i style='font-size:12px; color:gray;'>Applied S1: Baseline balanced needs.</i>"

    update_dashboard(False)

def export_report_txt(b):
    with open("ERP_Selection_Report.txt", "w") as f: f.write(last_report_txt)
    files.download("ERP_Selection_Report.txt")

def export_report_png(b):
    if global_fig is not None:
        global_fig.savefig("ERP_DSS_Charts.png", bbox_inches='tight', dpi=300, facecolor='white')
        files.download("ERP_DSS_Charts.png")

def export_report_csv(b):
    df = pd.DataFrame(working_scores).T
    df.columns = criteria
    df.to_csv("ERP_Raw_Data.csv")
    files.download("ERP_Raw_Data.csv")

def trigger_sensitivity(b):
    update_dashboard(show_sensitivity=True)

# Wiring
erp_selector.observe(on_erp_change, names='value')
slider_erps.observe(lambda x: update_dashboard(False), 'value')
for sl in sliders_w: sl.observe(lambda x: update_dashboard(False), 'value')
btn_s1.on_click(load_scenario); btn_s2.on_click(load_scenario); btn_s3.on_click(load_scenario); btn_s_avg.on_click(load_scenario)

btn_apply_scores.on_click(lambda x: [working_scores.update({erp_selector.value: [max(1, min(5, s.value)) for s in score_inputs]}), update_dashboard(False)])

btn_reset.on_click(reset_scores); btn_export_txt.on_click(export_report_txt); btn_export_png.on_click(export_report_png); btn_export_csv.on_click(export_report_csv)
btn_sens.on_click(trigger_sensitivity)

# Init
on_erp_change({'new': 'ERP-A'}); update_dashboard(False)

# === 6. LAYOUT ===
display(HTML("<h2 style='color:#2c3e50; font-family:Arial;'>Automated & Explainable ERP Pre-Selection (DSS)</h2>"))
ui_header = widgets.HBox([
    widgets.VBox([widgets.HTML("<b>1. Alternatives:</b>"), slider_erps]),
    widgets.VBox([
        widgets.HTML("<b>2. Scenarios:</b>"),
        widgets.HBox([btn_s1, btn_s2, btn_s3, btn_s_avg]),
        scenario_desc,
        widgets.HTML("<div style='margin-top:5px; max-width:550px;'><small><i><b>Methodological note:</b> The simulation uses a <b>single</b> aggregated raw score matrix. Diverse stakeholder opinions (e.g., Business vs. IT) are simulated exclusively through the weight scenarios above, not by altering the scores.</i></small></div>")
    ])
])
ui_scores = widgets.VBox([
    widgets.HTML("<br><b>4. Edit raw scores (1-5):</b><br><small><i><b>Methodological note:</b> Use discrete integers only. Qualitative criteria are evaluated on discrete scales; using decimals introduces 'false precision'.</i></small>"),
    widgets.HBox([erp_selector] + score_inputs + [btn_apply_scores])
])

ui_actions = widgets.VBox([
    widgets.HBox([btn_export_txt, btn_export_png, btn_export_csv]),
    widgets.HBox([btn_sens, btn_reset])
])

display(ui_header, widgets.HTML("<br><b>3. Criteria weights:</b>"), widgets.VBox(sliders_w), weight_sum_label, ui_scores,
        widgets.HTML("<br><b>5. Analytical tools & Data export:</b>"), ui_actions,
        widgets.HTML("<br><b>6. Interactive dashboard & Results:</b>"), out)

HTML(value='<br><b>3. Criteria weights:</b>')

HTML(value="<span style='color:green; font-size:13px;'>Sum: <b>1.00</b> (Perfect)</span>")

HTML(value='<br><b>5. Analytical tools & Data export:</b>')

HTML(value='<br><b>6. Interactive dashboard & Results:</b>')

Output()

In [16]:
# @title
